# 🔬 Lab 10 — Cryptanalysis of MT19937 PRNG Misuse
### Course: Cryptography Laboratory (22CPP307)
### Group: 01 | Nishant (2024UCP1773) & Lokesh Saini (2024UCP1505)

---

## 1. Executive Summary & Experimental Objectives
In **Lab 9**, we proved that non-cryptographic PRNGs like **Mersenne Twister (MT19937)** produce near-ideal statistical distributions while providing zero cryptographic security.

**Lab 10 demonstrates active exploitation of systems that misuse MT19937:**
1. **Part A — MT19937 Toy Stream Cipher Brute-Force**:
   - Recovering an unknown 16-bit seed (65,536 search space) using known plaintext bytes (14 consecutive 'A's) positioned at an unknown offset.
2. **Part B — Predictable Password Reset Token Cracking**:
   - Recovering the exact server timestamp and predicting 128-bit reset tokens within a search window of T +/- 300 seconds.
3. **Mathematical Foundations & Defensive Engineering**:
   - Analysis of search space reduction, false positive probabilities, and CSPRNG remediation using secrets.


In [ ]:
import os
import sys
import time
import random
import matplotlib.pyplot as plt

# Connect to local src modules
src_dir = os.path.abspath('../src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from stream_cipher import (
    generate_keystream, encrypt, decrypt,
    generate_vulnerable_plaintext, generate_16bit_seed
)
from seed_recovery_attack import brute_force_16bit_seed
from token_generator import generate_reset_token
from token_prediction_attack import predict_token

print("[+] Environment initialized successfully.")
print("[+] Python Version:", sys.version.split()[0])


---
## 2. Part A: MT19937 Toy Stream Cipher Known-Plaintext Attack

### Mathematical Formulation
The stream cipher encrypts a plaintext sequence P with keystream K:
$$C_i = P_i \oplus K_i, \quad K_i = \text{MT19937}(S) \pmod{256}$$

Because XOR is self-inverting:
$$P_i = C_i \oplus K_i$$

### Attack Algorithm
Given an intercepted ciphertext C containing a known sequence b"AAAAAAAAAAAAAA" at an unknown offset:
1. Iterate candidate seed s in [0, 65535].
2. Generate candidate keystream K of length |C|.
3. Decrypt candidate plaintext P = C XOR K.
4. Test if b"AAAAAAAAAAAAAA" is a substring of P.


In [ ]:
# Run Part A Simulation and Attack
known_seq = b"AAAAAAAAAAAAAA"
pt_str, prefix_str, prefix_len = generate_vulnerable_plaintext(min_prefix=5, max_prefix=20)
secret_seed = generate_16bit_seed()
ciphertext = encrypt(pt_str.encode('ascii'), secret_seed)

print("[*] Target Payload Generated:")
print(f"    Plaintext        : '{pt_str}' (Prefix: '{prefix_str}', Length: {prefix_len})")
print(f"    Secret 16b Seed  : {secret_seed} (0x{secret_seed:04X})")
print(f"    Ciphertext (Hex) : {ciphertext.hex()}")

print("\n[*] Launching Brute-Force Attack across 65,536 seeds...")
t0 = time.time()
attack_res = brute_force_16bit_seed(ciphertext, known_seq)
elapsed = time.time() - t0

print(f"[+] Attack Completed in {elapsed:.3f} seconds!")
print(f"    Recovered Seed   : {attack_res['recovered_seed']} ({attack_res['recovered_seed_hex']})")
print(f"    Match Confirmed  : {'YES [PASS]' if attack_res['recovered_seed'] == secret_seed else 'FAILED'}")
print(f"    Recovered Text   : '{attack_res['recovered_plaintext_str']}'")
print(f"    Known Offset     : Byte index {attack_res['known_sequence_offset']}")
print(f"    Throughput Rate  : {attack_res['seeds_per_second']:,.0f} seeds/second")


---
## 3. Part B: Predictable Password Reset Token Cracking

### The Vulnerability Pattern
Many legacy web applications generate tokens using:
```python
random.seed(int(time.time()))
token = bytes(random.getrandbits(8) for _ in range(16)).hex()
```

### Search Space Reduction
While a 16-byte token represents 2^128 states, seeding by Unix timestamp collapses the entropy:
$$|\mathcal{S}| = (T_{\text{approx}} + 300) - (T_{\text{approx}} - 300) + 1 = 601 \text{ candidates}$$
$$\text{Effective Entropy} = \log_2(601) \approx 9.23 \text{ bits}$$


In [ ]:
# Run Part B Simulation and Attack
client_time = int(time.time())
server_time = client_time + 42  # 42-second latency
issued_t, raw_tok, hex_tok = generate_reset_token(server_time)

print("[*] Victim Action:")
print(f"    Target Token (Hex) : {hex_tok}")
print(f"    Server Timestamp   : {server_time}")
print(f"    Client Timestamp   : {client_time}")

print("\n[*] Attacker Scanning Window [T_0 - 300, T_0 + 300]...")
pred_res = predict_token(hex_tok, client_time, window_seconds=300)

print(f"[+] Token Prediction Completed in {pred_res['elapsed_seconds']*1000.0:.2f} ms!")
print(f"    Recovered Time     : {pred_res['recovered_timestamp']} ({pred_res['recovered_timestamp_hex']})")
print(f"    Accuracy Match     : {'YES [PASS]' if pred_res['recovered_timestamp'] == server_time else 'FAILED'}")
print(f"    Time Offset        : {pred_res['time_offset_seconds']:+d} seconds")
print(f"    Candidates Tested  : {pred_res['candidates_tested']} / {pred_res['total_window_candidates']}")
print(f"    Regenerated Check  : {pred_res['regenerated_token']}")


---
## 4. Visualizations & Comparative Performance
Below we compare the empirical execution times and search space sizes of both attack vectors.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Search Space Comparison (Log Scale)
vectors = ['Part A (16-bit Seed)', 'Part B (Timestamp Window)', 'AES-128 / Secure Token']
spaces = [65536, 601, 2**128]
colors = ['cornflowerblue', 'crimson', 'darkgreen']

axes[0].bar(vectors, [16, 9.23, 128], color=colors, width=0.5)
axes[0].set_ylabel("Effective Entropy (Bits)", fontsize=11)
axes[0].set_title("Entropy Reduction: Vulnerable vs Secure", fontsize=12)
axes[0].set_ylim(0, 140)
for i, v in enumerate([16, 9.23, 128]):
    axes[0].text(i, v + 3, f"{v} bits", ha='center', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Subplot 2: Execution Latency (Milliseconds)
attack_names = ['Part A\n(65,536 seeds)', 'Part B\n(601 timestamps)']
latencies_ms = [attack_res['elapsed_seconds'] * 1000.0, pred_res['elapsed_seconds'] * 1000.0]

axes[1].bar(attack_names, latencies_ms, color=['royalblue', 'tomato'], width=0.45)
axes[1].set_ylabel("Execution Latency (ms)", fontsize=11)
axes[1].set_title("Attack Execution Latency", fontsize=12)
for i, v in enumerate(latencies_ms):
    axes[1].text(i, v * 1.05, f"{v:.1f} ms", ha='center', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


---
## 5. False Positive Probability Analysis

### Theorem: Substring Match Inversion
For a random candidate keystream K of length L and known target sequence M of length m = 14 bytes:
$$P(\text{single byte match}) = \frac{1}{256} = 2^{-8}$$
$$P(14 \text{ contiguous bytes match at a fixed offset}) = \left(\frac{1}{256}\right)^{14} = 2^{-112} \approx 1.93 \times 10^{-34}$$

Accounting for the sliding window across (L - m + 1) <= 20 possible offsets and all 65,536 candidate seeds:
$$P(\text{any false positive}) \le 20 \times 2^{16} \times 2^{-112} = 20 \times 2^{-96} \approx 2.5 \times 10^{-28}$$

**Conclusion**: The probability of encountering a false positive is mathematically zero, guaranteeing that any candidate seed yielding 14 consecutive 'A's is uniquely and unconditionally the true encryption seed.


---
## 6. Secure Defensive Countermeasures
To eliminate both vulnerabilities completely:
1. **Never use MT19937 for Cryptography**: MT19937 is explicitly designed for numerical simulation, not security.
2. **Use OS CSPRNG for Tokens**:
```python
import secrets

# Generates 256 bits of true cryptographic entropy from OS kernel CSPRNG
secure_reset_token = secrets.token_hex(32)
```
3. **Use Authenticated Encryption (AEAD)**: Replace toy stream ciphers with AES-256-GCM or ChaCha20-Poly1305.
